<a href="https://colab.research.google.com/github/1900690/cucumber/blob/main/fastlabel-ultraliticshub-boundingbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#zipを解凍
import shutil
import os

file_name ="cucanber-fruit-flower_20260410140249.zip"

#データを解凍
shutil.unpack_archive('/content/'+file_name, '/content/')
#zipを消す
os.remove('/content/'+file_name)

In [10]:
# @title データセット構成・リサイズ・全パターンオーグメンテーション
import os
import yaml
import shutil
import cv2
import numpy as np
import albumentations as A
import itertools
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# --- パスと分割の設定 ---
# @markdown ### フォルダパス
source_img_dir = '/content/original' # @param {type:"string"}
source_label_dir = '/content/yolo/annotations' # @param {type:"string"}
classes_file = '/content/yolo/classes.txt' # @param {type:"string"}
output_root_dir = '/content/dataset/detect' # @param {type:"string"}

# @markdown ### 分割比率とリサイズ
train_ratio = 0.7 # @param {type:"slider", min:0, max:1, step:0.1}
val_ratio = 0.2 # @param {type:"slider", min:0, max:1, step:0.1}
test_ratio = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}
max_pixel_size = 1280 # @param {type:"integer"}

# --- オーグメンテーション設定 ---
# @markdown ### 適用対象の選択
apply_to_train = True # @param {type:"boolean"}
apply_to_val = False # @param {type:"boolean"}
apply_to_test = False # @param {type:"boolean"}

# @markdown ### 加工メニュー (チェックした項目の全組み合わせを生成)
use_flip = True # @param {type:"boolean"}
use_rotation = False # @param {type:"boolean"}
use_translation = False # @param {type:"boolean"}
use_color_jitter = True # @param {type:"boolean"}
use_noise = True # @param {type:"boolean"}
use_blur = True # @param {type:"boolean"}

def get_active_transforms():
    """チェックされた加工とその識別名をリストで返す"""
    available = []
    if use_flip: available.append(('flip', A.HorizontalFlip(p=1.0)))
    if use_rotation: available.append(('rot', A.Rotate(limit=30, p=1.0)))
    if use_translation: available.append(('trans', A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=1.0)))
    if use_color_jitter: available.append(('color', A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0)))
    if use_noise: available.append(('noise', A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)))
    if use_blur: available.append(('blur', A.Blur(blur_limit=3, p=1.0)))
    return available

def resize_image(image, max_size):
    h, w = image.shape[:2]
    if max(h, w) <= max_size:
        return image
    scale = max_size / max(h, w)
    return cv2.resize(image, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

def load_labels(path):
    bboxes, labels = [], []
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    labels.append(int(parts[0]))
                    bboxes.append(list(map(float, parts[1:])))
    return bboxes, labels

def save_labels(path, bboxes, labels):
    with open(path, 'w') as f:
        for label, bbox in zip(labels, bboxes):
            f.write(f"{label} {' '.join([f'{x:.6f}' for x in bbox])}\n")

def process_dataset():
    if abs((train_ratio + val_ratio + test_ratio) - 1.0) > 1e-5:
        print("Error: 分割比率の合計を1.0にしてください。")
        return

    with open(classes_file, 'r') as f:
        class_names = [line.strip() for line in f if line.strip()]

    if os.path.exists(output_root_dir):
        shutil.rmtree(output_root_dir)

    all_images = [f for f in os.listdir(source_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    train_files, temp_files = train_test_split(all_images, test_size=(1 - train_ratio), random_state=42)
    val_test_sum = val_ratio + test_ratio
    val_files, test_files = train_test_split(temp_files, test_size=(test_ratio/val_test_sum), random_state=42) if val_test_sum > 0 else ([], [])

    splits = {'train': (train_files, apply_to_train), 'val': (val_files, apply_to_val), 'test': (test_files, apply_to_test)}

    # 有効な加工の全組み合わせを取得
    active_transforms = get_active_transforms()

    print(f"Processing started. Active combinations: {2**len(active_transforms)}")

    for split_name, (files, do_aug) in splits.items():
        if not files: continue
        img_dir = os.path.join(output_root_dir, 'images', split_name)
        lbl_dir = os.path.join(output_root_dir, 'labels', split_name)
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        for filename in tqdm(files, desc=f"Split: {split_name}"):
            basename = os.path.splitext(filename)[0]
            image = cv2.imread(os.path.join(source_img_dir, filename))
            if image is None: continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = resize_image(image, max_pixel_size)
            bboxes, labels = load_labels(os.path.join(source_label_dir, basename + '.txt'))

            # 全パターンの生成 (組み合わせ)
            # 例: [True, False] の組み合わせを全transform分作成
            patterns = list(itertools.product([False, True], repeat=len(active_transforms)))

            for p_idx, pattern in enumerate(patterns):
                # 適用する加工を抽出
                selected_transforms = [t[1] for i, t in enumerate(active_transforms) if pattern[i]]
                selected_names = [t[0] for i, t in enumerate(active_transforms) if pattern[i]]

                # 適用対象外のSplitで、かつ何らかの加工が選択されている場合はスキップ（オリジナルのみ保存）
                if not do_aug and any(pattern):
                    continue

                suffix = "_" + "_".join(selected_names) if selected_names else "_original"
                save_name = f"{basename}{suffix}"

                try:
                    # Composeして適用
                    aug_pipeline = A.Compose(selected_transforms, bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
                    res = aug_pipeline(image=image, bboxes=bboxes, class_labels=labels)

                    # 保存 (Bboxが空でない、または元々空の場合)
                    if len(res['bboxes']) == len(bboxes) or len(bboxes) == 0:
                        cv2.imwrite(os.path.join(img_dir, f"{save_name}.jpg"), cv2.cvtColor(res['image'], cv2.COLOR_RGB2BGR))
                        save_labels(os.path.join(lbl_dir, f"{save_name}.txt"), res['bboxes'], labels)
                except Exception:
                    continue

    # YAML
    yaml_dict = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {i: n for i, n in enumerate(class_names)}}
    with open(os.path.join(output_root_dir, 'dataset.yaml'), 'w') as f:
        yaml.dump(yaml_dict, f, default_flow_style=False, sort_keys=False)

    print(f"Done. Output: {output_root_dir}")

process_dataset()

Processing started. Active combinations: 16


/tmp/ipykernel_2667/3095114498.py:46: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  if use_noise: available.append(('noise', A.GaussNoise(var_limit=(10.0, 50.0), p=1.0)))


Split: train:   0%|          | 0/18 [00:00<?, ?it/s]

Split: val:   0%|          | 0/6 [00:00<?, ?it/s]

Split: test:   0%|          | 0/3 [00:00<?, ?it/s]

Done. Output: /content/dataset/detect


In [13]:
# @title  ZIP圧縮
import shutil
import os
from google.colab import files

# 圧縮したいフォルダのパス（前のセルで指定した output_root_dir）
target_dir = '/content/dataset' # @param {type:"string"}
# 作成するZIPファイル名（拡張子なし）
zip_filename = 'cucumber' # @param {type:"string"}

if os.path.exists(target_dir):
    print(f" {target_dir} を圧縮しています...")

    # フォルダをZIP圧縮
    # shutil.make_archive(出力名, 形式, 圧縮したいディレクトリ)
    shutil.make_archive(zip_filename, 'zip', target_dir)

    zip_path = f"{zip_filename}.zip"
    print(f" 圧縮完了: {zip_path}")

    # ダウンロード開始
    #print(" ダウンロードを開始します...")
    #files.download(zip_path)
else:
    print(f" エラー: {target_dir} が見つかりません。前のセルを先に実行してください。")

 /content/dataset を圧縮しています...
 圧縮完了: cucumber.zip


In [14]:
# ダウンロード開始
#print(" ダウンロードを開始します...")
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#魚眼補正

In [ ]:
# @title 魚眼キャリブレーション設定
# @markdown チェッカーボードのパスとパラメータを指定してください。

import cv2
import numpy as np
import os
import glob

# --- フォーム入力項目 ---
# 画像が保存されているディレクトリのパス（ワイルドカード指定）
image_path = '/content/checkboad/*.jpg' # @param {type:"string"}
# チェッカーボードの内側の角の数（列数）
board_width = 6 # @param {type:"integer"}
# チェッカーボードの内側の角の数（行数）
board_height = 9 # @param {type:"integer"}
# -----------------------

def run_calibration(img_glob_path, pattern_size):
    """
    魚眼レンズのキャリブレーションを実行し、カメラ行列Kと歪み係数Dを算出します。
    """
    # チェッカーボードの設定
    CHECKERBOARD = pattern_size
    subpix_criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)
    calibration_flags = (cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC +
                         cv2.fisheye.CALIB_CHECK_COND +
                         cv2.fisheye.CALIB_FIX_SKEW)

    # 3D空間上の点（オブジェクトポイント）の準備
    objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[0, :, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

    objpoints = [] # 3D points in real world space
    imgpoints = [] # 2d points in image plane
    _img_shape = None

    images = glob.glob(img_glob_path)

    if not images:
        print(f"Error: 画像が見つかりませんでした。パスを確認してください: {img_glob_path}")
        return None

    print(f"Processing: {len(images)} 枚の画像を解析中...")

    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue

        if _img_shape is None:
            _img_shape = img.shape[:2]
        elif _img_shape != img.shape[:2]:
            print(f"Skip: 画像サイズが一致しないためスキップしました: {fname}")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # チェッカーボードの角を検出
        ret, corners = cv2.findChessboardCorners(
            gray,
            CHECKERBOARD,
            cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE
        )

        if ret:
            objpoints.append(objp)
            cv2.cornerSubPix(gray, corners, (3, 3), (-1, -1), subpix_criteria)
            imgpoints.append(corners)
            print(f"Success: {os.path.basename(fname)}")

    num_valid_images = len(objpoints)
    if num_valid_images < 4:
        print(f"Error: 有効な画像が不足しています（現在 {num_valid_images} 枚）。")
        return None

    # 出力用変数の初期化
    K = np.zeros((3, 3))
    D = np.zeros((4, 1))
    rvecs = [np.zeros((1, 1, 3), dtype=np.float64) for _ in range(num_valid_images)]
    tvecs = [np.zeros((1, 1, 3), dtype=np.float64) for _ in range(num_valid_images)]

    # 魚眼レンズ用キャリブレーションの実行
    rms, _, _, _, _ = cv2.fisheye.calibrate(
        objpoints,
        imgpoints,
        gray.shape[::-1],
        K,
        D,
        rvecs,
        tvecs,
        calibration_flags,
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
    )

    # 結果出力
    print("\n--- Calibration Results ---")
    print(f"有效画像数: {num_valid_images}")
    print(f"再投影誤差 (RMS): {rms:.6f}")
    print(f"DIM = {_img_shape[::-1]}")
    print(f"K = np.array({K.tolist()})")
    print(f"D = np.array({D.tolist()})")
    print("---------------------------\n")

    return K, D, _img_shape[::-1]

# 実行
result = run_calibration(image_path, (board_width, board_height))

In [ ]:
# @title 魚眼画像補正の実行設定
# @markdown キャリブレーション結果の値を入力し、補正したい画像フォルダを指定してください。

import cv2
import numpy as np
import os
import glob

# --- フォーム入力項目 ---
# 補正したい画像が入っているフォルダのパス（例: /content/drive/MyDrive/input/）
input_folder = '/content/drive/MyDrive/fisheye_images/' # @param {type:"string"}
# 補正後の画像を保存するフォルダ
output_folder = '/content/drive/MyDrive/undistorted_images/' # @param {type:"string"}

# キャリブレーション結果を入力（先ほどのコードの出力をコピーしてください）
DIM = (1920, 1080) # @param {type:"raw"}
K = np.array([[560.12, 0.0, 960.0], [0.0, 560.12, 540.0], [0.0, 0.0, 1.0]]) # @param {type:"raw"}
D = np.array([[-0.05], [0.01], [-0.02], [0.01]]) # @param {type:"raw"}

# 補正の強さ調整 (0.0: 全てのピクセルを保持, 1.0: 歪みのない領域を最大化)
balance = 1.0 # @param {type:"slider", min:0, max:1, step:0.1}
# -----------------------

def undistort_folder(input_path, output_path, K, D, DIM, balance):
    # 保存先フォルダが存在しない場合は作成
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Directory created: {output_path}")

    # 画像リストの取得
    images = glob.glob(os.path.join(input_path, "*.jpg")) + glob.glob(os.path.join(input_path, "*.png"))

    if not images:
        print(f"Error: 画像が見つかりませんでした: {input_path}")
        return

    # 補正用マップの作成（一度だけ計算）
    # balance=0.0 は元の解像度重視、balance=1.0 は黒い縁をなくす方向に調整
    new_K = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
        K, D, DIM, np.eye(3), balance=balance
    )
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(
        K, D, np.eye(3), new_K, DIM, cv2.CV_16SC2
    )

    print(f"Starting processing: {len(images)} files")

    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue

        # 補正（再マッピング）の実行
        undistorted_img = cv2.remap(
            img, map1, map2,
            interpolation=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT
        )

        # 保存
        base_name = os.path.basename(fname)
        save_path = os.path.join(output_path, "undistorted_" + base_name)
        cv2.imwrite(save_path, undistorted_img)
        print(f"Saved: {save_path}")

    print("\nProcessing complete.")

# 実行
undistort_folder(input_folder, output_folder, K, D, DIM, balance)